# 6. CLI workflows

The CLI is the easiest way to learn the complete local lifecycle because every state-changing action returns structured JSON and every durable object has a read command. Run these examples in a disposable directory with a real pipeline catalog installed.


In [ ]:
from provium_pipeline.cli import CLIResult, LocalCLIBackend

assert CLIResult and LocalCLIBackend
groups = ('pipeline', 'input-set', 'run', 'dispatch')
groups


## 1. Inspect and validate intent

```bash
provium pipeline list
provium pipeline validate pipeline.yaml
provium pipeline show pipeline.yaml
```

Validation performs definition loading, catalog resolution, configuration resolution, and compilation without creating a run. Fix all diagnostics here first.


## 2. Choose an input path

Stored NDJSON input set:

```bash
provium input-set create records.ndjson --identifier tutorial.batch
provium input-set show INPUT_SET_ID
provium input-set export INPUT_SET_ID --output frozen-records.ndjson
provium input-set artifacts INPUT_SET_ID
```

Resolver-backed inputs avoid writing NDJSON manually:

```bash
provium pipeline enqueue pipeline.yaml --records-from RESOLVER_SPEC
```

Both paths freeze an immutable snapshot before task planning.


## 3. Create, inspect, and execute

```bash
provium run create pipeline.yaml --input-set tutorial.batch
provium run status RUN_ID
provium run inputs RUN_ID
provium run tasks RUN_ID
provium run execute RUN_ID
provium run outputs RUN_ID
provium run artifacts RUN_ID
```

`pipeline execute` is the convenience form that resolves inputs, creates the run, creates the dispatch, works it locally, and returns the result. Use the separate commands while learning because each boundary remains visible.


## 4. Partial work, observation, retry, and cancellation

```bash
provium run execute RUN_ID --node tokenize
provium run execute RUN_ID --record chapter-2
provium dispatch show DISPATCH_ID
provium dispatch wait DISPATCH_ID
provium dispatch retry DISPATCH_ID
provium run cancel RUN_ID
```

Selection filters tasks; they do not supply artifact content. Dependency policy determines whether missing upstream work is included or rejected. Retry creates new attempts for eligible work, preserving the audit trail.


## 5. Export evidence

```bash
provium run configuration export RUN_ID --output resolved-config.json
provium run export RUN_ID --output run-bundle.json
```

The bundle contains the frozen definition/configuration/input facts, run/task/output state, and audit timeline. Sensitive-looking configuration keys are redacted. Use exports for review and debugging, not as a replacement for the artifact store.


In [ ]:
expected_read_surfaces = {
    'run': {'status', 'tasks', 'inputs', 'outputs', 'artifacts'},
    'dispatch': {'show', 'wait'},
    'input-set': {'show', 'export', 'artifacts'},
}
assert 'outputs' in expected_read_surfaces['run']
expected_read_surfaces


**What to notice:** use returned IDs rather than guessing state, inspect after each mutation, and keep `.provium` as durable operational state. Next: [extension points and testing](07-extension-points-and-testing.ipynb). Reference: [CLI reference](../docs/cli.md) and [operations](../docs/operations.md).
